# E02 — o orçamento do vigia

O capítulo anterior deixou um instrumento e um incômodo. O instrumento conta quantas vezes o
preço rompeu a linha dentro de uma janela móvel de 60 dias; o incômodo é que os blocos têm
data, e a média não vê a data. Este caderno transforma o instrumento em **vigia** e mede o
preço de ligá-lo.

**O que se mede.** Um alarme são duas afirmações ao mesmo tempo, e medir uma só engana:

1. **com que frequência ele soa quando nada muda** — mil mundos sorteados, sem mudança
   nenhuma, submetidos à mesma rotina;
2. **quanto ele demora quando algo muda** — os alarmes do mundo real, contra o topo e o fundo
   de cada tombo: quanto do prejuízo já estava pago no dia do aviso.

**Duas leituras do mesmo corte.** O corte do capítulo anterior admite dois vigias, e o caderno
mede os dois: **contar** as violações em blocos e alarmar no limiar (a leitura do capítulo
anterior, cujo orçamento só se sabe medindo), e **aprofundar** o corte e alarmar quando o
próprio dia fica abaixo do k-ésimo pior (leitura cujo orçamento é exato, k/(n+1), e por isso
tem limite provado).

**E a forma da mudança.** A latência não depende só do orçamento: o mesmo vigia, com o mesmo
orçamento, chega em dias quando a cauda se abre de uma vez e em meses quando a média se
desloca. Por isso o caderno fecha medindo três formas de mudança em mundos onde a data da
mudança é conhecida.

**Convenções** (AGENTS.md §7 e §9): parâmetros no topo marcados "brinque com", algoritmo em
frevolab, resultado em lab/resultados/E02_orcamento.json, figura em .pdf e .png.

In [1]:
# <- brinque com: SERIE, JANELA, CAUDA, BLOCO, LIMIAR, MUNDOS, LIMIARES, SEMENTE,
#                JANELAS_DECLARADAS, MUNDOS_PARADO, MUNDO_DIAS, MUDANCA_EM, REPETICOES
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, mudanca, promessa, vigia, volatilidade

RAIZ = Path.cwd()
SERIE = "sp500.csv"
JANELA = 252            # o corte do capitulo anterior
CAUDA = 0.05
BLOCO = 60              # o bloco em que as violacoes sao contadas
LIMIAR = 13             # o teto que nenhum dos 200 mundos anteriores passou
MUNDOS = 1000           # mundos que nunca mudam, para calibrar o falso alarme
LIMIARES = (8, 9, 10, 11, 12, 13)   # a varredura da troca
POR_EXTENSO = {8: "oito", 9: "nove", 10: "dez", 11: "onze", 12: "doze", 13: "treze"}
SEMENTE = 23
JANELA_CINCO_ANOS = 1260
JANELAS_DECLARADAS = (21, 63, 252, JANELA_CINCO_ANOS)  # a segunda leitura: a memoria que compra o orcamento
MUNDOS_PARADO = 200     # mundos que nunca mudam, para conferir o orcamento declarado
MUNDO_DIAS = 3000       # o tamanho do mundo sintetico em que a latencia e medida
MUDANCA_EM = 1500       # o dia em que a mudanca entra nesse mundo
REPETICOES = 200        # quantos mundos sorteados por forma de mudanca

retornos = volatilidade.retornos_log(dados.carregar_serie(SERIE))
preco = dados.carregar_serie(SERIE).loc[retornos.index]
violacoes = promessa.violacoes(retornos, JANELA, CAUDA)
contagem = promessa.conta_em_blocos(violacoes, BLOCO)
print("frevolab %s | %s: %d pregoes | blocos medidos: %d | limiar: %d" % (
    frevolab.VERSAO, SERIE, len(retornos), len(contagem), LIMIAR))

frevolab 0.1.0 | sp500.csv: 6718 pregoes | blocos medidos: 6407 | limiar: 13


## Quanto ele soa quando nada muda

In [2]:
# O mundo que nunca muda, submetido a mesma rotina: e com que frequencia ele soa sozinho.
sorteio = np.random.default_rng(SEMENTE)
blocos_nulos = np.empty((MUNDOS, len(contagem)))
for i in range(MUNDOS):
    mundo = pd.Series(sorteio.normal(0.0, 0.01, len(retornos)), index=retornos.index)
    blocos_nulos[i] = promessa.conta_em_blocos(promessa.violacoes(mundo, JANELA, CAUDA), BLOCO)

orcamentos = {t: vigia.orcamento(blocos_nulos, t) for t in LIMIARES}
probabilidade = float(promessa.entrega_do_corte(JANELA, CAUDA))

print("mundo parado: %d mundos de %.1f anos cada" % (
    MUNDOS, orcamentos[LIMIAR]["anos_por_mundo"]))
print()
print("%6s %12s %10s %12s %14s %14s" % (
    "limiar", "dias/mundo", "anos/soa", "mundos soam", "teto iid (%)", "real: anos/soa"))
for t in LIMIARES:
    o = orcamentos[t]
    n_real = len(vigia.alarmes(contagem, t))
    anos_reais = len(contagem) / 252.0
    teto = 100 * vigia.teto_independente(len(contagem), BLOCO, probabilidade, t)
    print("%6d %12.4f %10.0f %11.1f%% %14.4f %14s" % (
        t, o["dias_por_mundo"], o["anos_por_alarme"], 100 * o["fracao_com_alarme"], teto,
        ("%.2f" % (anos_reais / n_real)) if n_real else "-"))

mundo parado: 1000 mundos de 25.4 anos cada

limiar   dias/mundo   anos/soa  mundos soam   teto iid (%) real: anos/soa
     8      67.2240          4        99.6%      7315.6146           1.41
     9      19.0470         13        78.1%      2187.4640           1.82
    10       5.0560         42        37.0%       582.5427           2.12
    11       1.1850        171        10.4%       139.1832           1.96
    12       0.2280        748         2.7%        30.0228           2.31
    13       0.0320       3632         0.6%         5.8786           3.63


## Quanto ele demora quando algo muda

In [3]:
# O mundo real: os alarmes do limiar escolhido, e o que ja estava pago em cada um.
episodios = vigia.alarmes(contagem, LIMIAR)
pagamentos = [vigia.prejuizo_pago(preco, e["inicio"]) for e in episodios]

print("limiar %d: %d alarmes em %.1f anos" % (LIMIAR, len(episodios), len(contagem) / 252.0))
print()
print("%12s %12s %9s %11s %11s %9s" % (
    "alarme", "topo", "atraso", "queda ate", "queda total", "ja pago"))
for e, p in zip(episodios, pagamentos):
    print("%12s %12s %6d d %10.1f%% %10.1f%% %8.0f%%" % (
        str(e["inicio"].date()), str(p["topo"].date()), p["atraso_dias"],
        100 * p["queda_ate_alarme"], 100 * p["queda_total"], 100 * p["fracao_paga"]))
fracoes = np.array([p["fracao_paga"] for p in pagamentos])
print()
print("mediana do prejuizo ja pago: %.0f%% | pior alarme: %.0f%%" % (
    100 * np.median(fracoes), 100 * fracoes.max()))

limiar 13: 7 alarmes em 25.4 anos

      alarme         topo    atraso   queda ate queda total   ja pago
  2002-09-19   2002-01-04    258 d       28.1%       33.8%       83%
  2007-08-14   2007-07-19     26 d        8.1%       21.8%       37%
  2008-10-22   2007-10-31    357 d       42.1%       56.3%       75%
  2011-09-30   2011-04-29    154 d       17.0%       19.4%       88%
  2018-04-02   2018-01-26     66 d       10.1%       18.2%       56%
  2019-01-03   2018-09-20    105 d       16.5%       16.5%      100%
  2020-03-18   2020-02-19     28 d       29.2%       33.9%       86%

mediana do prejuizo ja pago: 83% | pior alarme: 100%


## O mesmo corte lido mais fundo

In [4]:
# O mesmo corte lido mais fundo: o posto que o orcamento declarado escolhe, em vez da contagem.
segunda, linhas = {}, []
for n in JANELAS_DECLARADAS:
    alfa = vigia.orcamento_minimo(n)      # o mais raro que n dias de memoria conseguem declarar
    a = vigia.dispara(retornos, n, alfa)
    datas = a.index[a.to_numpy()]
    d20 = [d for d in datas if d.year == 2020]
    d08 = [d for d in datas if d.year == 2008]
    segunda[n] = {"d20": d20[0] if d20 else None, "d08": d08[0] if d08 else None,
                  "alarmes": int(a.sum()), "por_ano": vigia.alarmes_por_ano(a)}
    linhas.append({
        "janela": n,
        "declara (al/ano)": 252 * alfa,
        "um alarme a cada (anos)": 1 / (252 * alfa),
        "alarmes no real": int(a.sum()),
        "alarmes/ano no real": vigia.alarmes_por_ano(a),
        "primeiro de 2020": str(d20[0].date()) if d20 else "-",
        "ja pago em 2020 (%)": 100 * vigia.prejuizo_pago(preco, d20[0])["fracao_paga"] if d20 else float("nan"),
    })
print(pd.DataFrame(linhas).round(3).to_string(index=False))
print()
print("memoria para um alarme por ano: %d dias | para um alarme por decada: %d dias" % (
    vigia.memoria_para(1 / 253), vigia.memoria_para(1 / 2521)))

# O orcamento declarado e entregue? A conferencia e a de sempre: mundos em que nada muda.
sorteio_declarado = np.random.default_rng(SEMENTE + 1)
nulos = [int(vigia.dispara(pd.Series(sorteio_declarado.normal(0.0, 0.01, len(retornos)),
                                    index=retornos.index),
                           JANELA, vigia.orcamento_minimo(JANELA)).sum())
         for _ in range(MUNDOS_PARADO)]
declarado_esperado = (len(retornos) - JANELA) * vigia.orcamento_minimo(JANELA)
print("mundo parado na janela de %d dias: %.1f +- %.1f alarmes | o orcamento declara %.1f" % (
    JANELA, np.mean(nulos), np.std(nulos, ddof=1), declarado_esperado))

 janela  declara (al/ano)  um alarme a cada (anos)  alarmes no real  alarmes/ano no real primeiro de 2020  ja pago em 2020 (%)
     21            11.455                    0.087              312               11.740       2020-01-03                2.254
     63             3.938                    0.254              113                4.279       2020-01-24                3.127
    252             0.996                    1.004               33                1.286       2020-02-24               13.951
   1260             0.200                    5.004               10                0.462       2020-02-27               35.464

memoria para um alarme por ano: 252 dias | para um alarme por decada: 2520 dias


mundo parado na janela de 252 dias: 25.7 +- 3.6 alarmes | o orcamento declara 25.6


## A forma decide o preço

In [5]:
# A forma da mudanca decide o preco: mesmo orcamento, mesma rotina, formas com data conhecida.
FORMAS = (("degrau", mudanca.degrau), ("rampa", mudanca.rampa), ("deriva", mudanca.deriva))
latencia_mediana, sem_alarme, falsos_antes = {}, {}, {}
for janela in (21, JANELA):
    for nome, gerador in FORMAS:
        sorteio_forma = np.random.default_rng(SEMENTE + 7)
        faltas, falsos = [], []
        for _ in range(REPETICOES):
            serie = pd.Series(gerador(MUNDO_DIAS, sorteio_forma))
            a = vigia.dispara(serie, janela, vigia.orcamento_minimo(janela))
            faltas.append(vigia.latencia(a, MUDANCA_EM))
            falsos.append(vigia.falsos_antes(a, MUDANCA_EM))
        faltas = np.array(faltas, dtype=float)
        latencia_mediana[(janela, nome)] = float(np.nanmedian(faltas))
        sem_alarme[(janela, nome)] = int(np.isnan(faltas).sum())
        falsos_antes[(janela, nome)] = float(np.mean(falsos))
        print("janela %4d %-7s: latencia mediana %5.0f dias | sem alarme %3d/%d | falsos antes %.2f"
              % (janela, nome, latencia_mediana[(janela, nome)],
                 sem_alarme[(janela, nome)], REPETICOES, falsos_antes[(janela, nome)]))

# Figura 3: o mesmo orcamento, tres formas, dois comprimentos de memoria.
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
x = np.arange(len(FORMAS))
for i, janela in enumerate((21, JANELA)):
    eixo.bar(x + (i - 0.5) * 0.38, [latencia_mediana[(janela, nome)] for nome, _ in FORMAS], 0.38,
             color=("#7f7f7f", "#1f4e79")[i],
             label="memoria de %d dias: %.1f alarmes por ano" % (janela, 252 * vigia.orcamento_minimo(janela)))
eixo.set_xticks(x)
eixo.set_xticklabels(["a cauda se abre de uma vez", "a cauda se aproxima devagar", "a media se desloca"])
eixo.set_ylabel("dias ate o primeiro alarme")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E02_orcamento", 3)
plt.close(fig)

janela   21 degrau : latencia mediana     3 dias | sem alarme   0/200 | falsos antes 67.17
janela   21 rampa  : latencia mediana    12 dias | sem alarme   0/200 | falsos antes 67.17


janela   21 deriva : latencia mediana    12 dias | sem alarme   0/200 | falsos antes 67.17


janela  252 degrau : latencia mediana     7 dias | sem alarme   0/200 | falsos antes 5.11


janela  252 rampa  : latencia mediana    69 dias | sem alarme   0/200 | falsos antes 5.11


janela  252 deriva : latencia mediana   150 dias | sem alarme   0/200 | falsos antes 5.11


## As figuras

In [6]:
# Figura 1: a troca. O que se compra e o que se paga, limiar por limiar.
anos_parado, pago_2020 = [], []
for t in LIMIARES:
    e2020 = [e for e in vigia.alarmes(contagem, t) if e["inicio"].year == 2020]
    if not e2020:
        continue
    anos_parado.append(orcamentos[t]["anos_por_alarme"])
    pago_2020.append(100 * vigia.prejuizo_pago(preco, e2020[0]["inicio"])["fracao_paga"])

fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.plot(anos_parado, pago_2020, "o-", color="#1f4e79", lw=1.6, ms=6)
for t, x, y in zip(LIMIARES, anos_parado, pago_2020):
    eixo.annotate("limiar %d" % t, (x, y), textcoords="offset points", xytext=(7, -4), fontsize=9)
eixo.set_xscale("log")
eixo.set_xlabel("um alarme falso a cada quantos anos de mundo parado")
eixo.set_ylabel("prejuizo ja pago quando o alarme soou em 2020 (%)")
eixo.set_ylim(0, 100)
eixo.grid(alpha=0.25)
graficos.salvar(fig, "E02_orcamento", 1)
plt.close(fig)
print("anos por alarme: %s" % [round(a) for a in anos_parado])
print("ja pago em 2020:  %s" % [round(p) for p in pago_2020])

anos por alarme: [4, 13, 42, 171, 748, 3632]
ja pago em 2020:  [36, 56, 56, 79, 87, 86]


## Leitura visual das figuras

Feita nesta sessão abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não
número.

**Figura 1.** O eixo x é logarítmico, de uns poucos anos a alguns milhares, e o y vai de 0 a
100. A curva sobe depressa no começo e achata no fim: os primeiros limiares compram muita
antecipação pagando pouco em falso alarme, e os últimos quase não movem o prejuízo já pago
enquanto custam três ordens de grandeza. Há um trecho plano no meio — dois limiares
vizinhos entregando exatamente a mesma coisa. O que o eixo engana: no logaritmo, o começo e
o fim da curva parecem vizinhos, e é justamente a distância entre eles que é o preço.

**Figura 2.** Dois painéis com o tempo em comum. Em cima, o índice: sobe até fevereiro, cai
quase vertical em março e se recupera depois. A linha do alarme cai no meio da queda, não
no começo dela. Embaixo, as violações nos 60 dias anteriores: ficam perto da promessa antes
e cruzam o limiar exatamente na linha do alarme. O que o painel de baixo mostra e o de cima
esconde é que a contagem sobe depois de o preço já ter descido.

**Figura 3.** Três grupos de duas barras, com o eixo até pouco mais de 140 dias. No grupo da
esquerda, a cauda que se abre de uma vez, as barras são baixas --- uns 3 e uns 7 dias. No
grupo do meio, a cauda que se aproxima devagar, a barra cinza continua baixa e a azul salta
para perto de 70. No grupo da direita, a média que se desloca, a cinza não se move e a azul
encosta no topo do gráfico. A legenda é o que impede a leitura errada: a cinza é o orçamento
falador, com 11,5 alarmes por ano, e a azul é o raro, com um por ano --- barra mais alta aqui
é alarme mais tardio, não vigia melhor. O que o eixo engana: as duas barras de cada grupo
parecem comparáveis, e só a legenda diz que foram declaradas em orçamentos diferentes; sem
ela, a figura sugeriria que a memória longa é pior, quando o que ela faz é comprar silêncio.


In [7]:
# Figura 2: o tombo de 2020, o alarme e o piso.
alarme = [e for e in episodios if e["inicio"].year == 2020][0]["inicio"]
pagamento = vigia.prejuizo_pago(preco, alarme)
de, ate = pd.Timestamp("2019-06-01"), pd.Timestamp("2020-12-31")

fig, (cima, baixo) = plt.subplots(2, 1, figsize=(9.4, 6.0), sharex=True,
                                  gridspec_kw={"height_ratios": [1.5, 1]})
cima.plot(preco.loc[de:ate].index, preco.loc[de:ate].to_numpy(), color="#1f4e79", lw=1.4)
cima.axvline(alarme, color="#b03a2e", lw=1.4, ls="--")
cima.annotate("alarme: %s\nja pagos %.0f%% dos %.0f%%" % (
    alarme.date(), 100 * pagamento["fracao_paga"], 100 * pagamento["queda_total"]),
    xy=(alarme, float(preco.loc[alarme])), xytext=(-190, 30), textcoords="offset points",
    fontsize=9, arrowprops=dict(arrowstyle="->", color="#b03a2e"))
cima.axvline(pagamento["topo"], color="#7f7f7f", lw=1.0, ls=":")
cima.set_ylabel("indice")
cima.grid(alpha=0.25)

baixo.plot(contagem.loc[de:ate].index, contagem.loc[de:ate].to_numpy(), color="#1f4e79", lw=1.4)
baixo.axhline(LIMIAR, color="#b03a2e", ls="--", lw=1.2, label="o limiar: %d" % LIMIAR)
baixo.axvline(alarme, color="#b03a2e", lw=1.4, ls="--")
baixo.set_ylabel("violacoes nos %d dias\nanteriores" % BLOCO)
baixo.set_xlabel("ano")
baixo.legend(frameon=False, fontsize=9)
baixo.grid(alpha=0.25)

fig.tight_layout()
graficos.salvar(fig, "E02_orcamento", 2)
plt.close(fig)
print("piso de atraso para o limiar %d: %d pregoes" % (LIMIAR, vigia.piso_de_atraso(LIMIAR)))
print("atraso medido em 2020: %d dias corridos" % pagamento["atraso_dias"])

piso de atraso para o limiar 13: 13 pregoes
atraso medido em 2020: 28 dias corridos


In [8]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
o13 = orcamentos[LIMIAR]
anos_reais = len(contagem) / 252.0
e2020 = [e for e in episodios if e["inicio"].year == 2020][0]["inicio"]
p2020 = vigia.prejuizo_pago(preco, e2020)
e2008 = [e for e in episodios if e["inicio"].year == 2008][0]["inicio"]
p2008 = vigia.prejuizo_pago(preco, e2008)

MESES = ("janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho",
         "agosto", "setembro", "outubro", "novembro", "dezembro")
rotulo = lambda d: "%d de %s de %d" % (d.day, MESES[d.month - 1], d.year)

resultado = {
    "vigia_limiar": LIMIAR,
    "vigia_bloco": BLOCO,
    "vigia_mundos": MUNDOS,
    "vigia_dias_parado_por_mundo": o13["dias_por_mundo"],
    "vigia_anos_parado_por_alarme": o13["anos_por_alarme"],
    "vigia_mundos_que_soam_pct": 100 * o13["fracao_com_alarme"],
    "vigia_teto_independente_pct": 100 * vigia.teto_independente(
        len(contagem), BLOCO, probabilidade, LIMIAR),
    "vigia_alarmes_reais": len(episodios),
    "vigia_anos_reais_por_alarme": anos_reais / len(episodios),
    "vigia_razao_real_contra_parado": o13["anos_por_alarme"] / (anos_reais / len(episodios)),
    "vigia_pago_mediano_pct": 100 * float(np.median(fracoes)),
    "vigia_pago_pior_pct": 100 * float(fracoes.max()),
    "vigia_piso_pregoes": vigia.piso_de_atraso(LIMIAR),
    "vigia_parado_alarmes": int(round(o13["episodios_por_mundo"] * MUNDOS)),
    "vigia_parado_erro_pct": 100.0 / np.sqrt(max(o13["episodios_por_mundo"] * MUNDOS, 1.0)),
    "vigia_tombo_recente_data": rotulo(e2020),
    "vigia_tombo_recente_atraso_dias": p2020["atraso_dias"],
    "vigia_tombo_recente_queda_pct": 100 * p2020["queda_ate_alarme"],
    "vigia_tombo_recente_total_pct": 100 * p2020["queda_total"],
    "vigia_tombo_recente_pago_pct": 100 * p2020["fracao_paga"],
    "vigia_tombo_antigo_data": rotulo(e2008),
    "vigia_tombo_antigo_atraso_dias": p2008["atraso_dias"],
    "vigia_tombo_antigo_pago_pct": 100 * p2008["fracao_paga"],
    "memoria_um_alarme_por_ano_dias": float(vigia.memoria_para(1 / 253)),
    "memoria_um_alarme_por_decada_dias": float(vigia.memoria_para(1 / 2521)),
    "fundo_um_ano_alarmes_por_ano": float(252 * vigia.orcamento_minimo(JANELA)),
    "fundo_um_mes_alarmes_por_ano": float(252 * vigia.orcamento_minimo(min(JANELAS_DECLARADAS))),
    "fundo_cinco_anos_alarmes_por_ano": float(252 * vigia.orcamento_minimo(JANELA_CINCO_ANOS)),
    "segunda_leitura_alarmes_reais": float(segunda[JANELA]["alarmes"]),
    "segunda_leitura_alarmes_ano": float(segunda[JANELA]["por_ano"]),
    "segunda_leitura_parado_alarmes": float(np.mean(nulos)),
    "segunda_leitura_parado_declarado": float(declarado_esperado),
    "segunda_leitura_tombo_recente_data": rotulo(segunda[JANELA]["d20"]),
    "segunda_leitura_tombo_recente_pago_pct": 100 * vigia.prejuizo_pago(
        preco, segunda[JANELA]["d20"])["fracao_paga"],
    "segunda_leitura_tombo_antigo_data": rotulo(segunda[JANELA]["d08"]),
    "segunda_leitura_tombo_antigo_pago_pct": 100 * vigia.prejuizo_pago(
        preco, segunda[JANELA]["d08"])["fracao_paga"],
    "segunda_leitura_cinco_anos_tombo_recente_data": rotulo(segunda[JANELA_CINCO_ANOS]["d20"]),
    "segunda_leitura_cinco_anos_tombo_recente_pago_pct": 100 * vigia.prejuizo_pago(
        preco, segunda[JANELA_CINCO_ANOS]["d20"])["fracao_paga"],
    "forma_degrau_dias": latencia_mediana[(JANELA, "degrau")],
    "forma_rampa_dias": latencia_mediana[(JANELA, "rampa")],
    "forma_deriva_dias": latencia_mediana[(JANELA, "deriva")],
    "forma_curta_degrau_dias": latencia_mediana[(21, "degrau")],
    "forma_curta_rampa_dias": latencia_mediana[(21, "rampa")],
    "forma_curta_deriva_dias": latencia_mediana[(21, "deriva")],
    "forma_repeticoes": float(REPETICOES),
    "forma_mundo_dias": float(MUNDO_DIAS),
    "forma_mudanca_dia": float(MUDANCA_EM),
}
for t, x, y in zip(LIMIARES, anos_parado, pago_2020):
    if t != LIMIAR:   # o limiar do capitulo ja esta em numVigiaLimiar; repetir medida e ruido
        resultado["troca_%s_limiar" % POR_EXTENSO[t]] = int(t)
    resultado["troca_%s_anos" % POR_EXTENSO[t]] = float(x)
    resultado["troca_%s_pago_pct" % POR_EXTENSO[t]] = float(y)

caminho = Path("lab/resultados/E02_orcamento.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E02_orcamento.json gravado | 64 grandezas
